# Week 7: AI Agents

**Lab companion to [week_07.md](../agenda/week_07.md)**

In this lab, you will:
1. Build a ReAct agent (Reason + Act)
2. Implement planning and execution
3. Handle multi-step tasks
4. Create an autonomous agent loop

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json
from typing import Callable

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. ReAct Pattern

Reason → Act → Observe → Repeat

In [ ]:
# Define available tools
def search(query: str) -> str:
    """Search for information."""
    mock_results = {
        "python": "Python is a programming language created in 1991.",
        "capital of france": "Paris is the capital of France.",
        "weather": "Current weather: 20°C, partly cloudy."
    }
    for key, value in mock_results.items():
        if key in query.lower():
            return value
    return f"No results found for: {query}"

def calculate(expression: str) -> str:
    """Calculate a math expression."""
    try:
        # Safe eval for simple math
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except:
        return "Error calculating expression"

def get_date() -> str:
    """Get current date."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

TOOLS = {
    "search": search,
    "calculate": calculate,
    "get_date": get_date
}

In [ ]:
REACT_PROMPT = """You are an AI agent that solves tasks step by step.

Available tools:
- search(query): Search for information
- calculate(expression): Calculate math expressions
- get_date(): Get current date

For each step, you MUST use this format:
Thought: <your reasoning about what to do>
Action: <tool_name>("<argument>")

After receiving an observation, continue with another Thought/Action.
When you have the final answer, use:
Thought: I have the answer
Final Answer: <your answer>

Task: {task}
"""

def run_react_agent(task: str, max_steps: int = 5) -> str:
    """Run ReAct agent."""
    messages = [
        {"role": "user", "content": REACT_PROMPT.format(task=task)}
    ]

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")

        # Get agent response
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages,
            stop=["Observation:"]  # Stop before observation
        )

        agent_output = response.choices[0].message.content
        print(agent_output)

        # Check for final answer
        if "Final Answer:" in agent_output:
            return agent_output.split("Final Answer:")[1].strip()

        # Parse action
        if "Action:" in agent_output:
            action_line = [l for l in agent_output.split("\n") if l.startswith("Action:")][0]
            # Parse: Action: tool_name("arg")
            import re
            match = re.search(r'Action:\s*(\w+)\("?([^"\)]*)"?\)', action_line)

            if match:
                tool_name = match.group(1)
                tool_arg = match.group(2)

                # Execute tool
                if tool_name in TOOLS:
                    if tool_arg:
                        observation = TOOLS[tool_name](tool_arg)
                    else:
                        observation = TOOLS[tool_name]()
                else:
                    observation = f"Unknown tool: {tool_name}"

                print(f"Observation: {observation}")

                # Add to messages
                messages.append({"role": "assistant", "content": agent_output})
                messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "Max steps reached without answer"

# Test
result = run_react_agent("What is the capital of France?")
print(f"\n✓ Final: {result}")

In [ ]:
# Multi-step task
result = run_react_agent("What's 25 * 4 + 50? Then search for Python.")
print(f"\n✓ Final: {result}")

## 2. OpenAI Native Tool Agent

In [ ]:
# Define tools in OpenAI format
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search for information on a topic",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_date",
            "description": "Get the current date",
            "parameters": {"type": "object", "properties": {}}
        }
    }
]

class Agent:
    """Simple agent using OpenAI tool calling."""

    def __init__(self, tools: list, tool_functions: dict):
        self.client = OpenAI()
        self.tools = tools
        self.tool_functions = tool_functions
        self.messages = []

    def run(self, task: str, max_iterations: int = 10) -> str:
        """Execute task with agent loop."""

        self.messages = [
            {"role": "system", "content": "You are a helpful agent. Use tools to complete tasks."},
            {"role": "user", "content": task}
        ]

        for i in range(max_iterations):
            response = self.client.chat.completions.create(
                model="gpt-5-mini",
                messages=self.messages,
                tools=self.tools
            )

            message = response.choices[0].message

            # No tool calls = done
            if not message.tool_calls:
                return message.content

            # Execute tools
            self.messages.append(message)

            for tool_call in message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                print(f"  🔧 {name}({args})")

                func = self.tool_functions[name]
                if args:
                    result = func(**args)
                else:
                    result = func()

                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

        return "Task incomplete"

# Create agent
agent = Agent(openai_tools, TOOLS)

# Test
result = agent.run("What is 100 / 4? Also, what date is today?")
print(f"\nResult: {result}")

## 3. Planning Agent

In [ ]:
class PlanningAgent:
    """Agent that plans before executing."""

    def __init__(self):
        self.client = OpenAI()
        self.tools = TOOLS

    def create_plan(self, task: str) -> list:
        """Create a plan of steps."""
        prompt = f"""Create a plan to accomplish this task.

Available tools: search, calculate, get_date

Task: {task}

Return a JSON array of steps:
[
    {{"step": 1, "action": "tool_name", "input": "tool input", "purpose": "why"}},
    ...
]

Only include steps that require tools. Max 5 steps."""

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )

        content = response.choices[0].message.content
        data = json.loads(content)

        # Handle both formats
        if isinstance(data, list):
            return data
        return data.get("steps", data.get("plan", []))

    def execute_plan(self, plan: list) -> list:
        """Execute each step in the plan."""
        results = []

        for step in plan:
            action = step.get("action", "")
            input_val = step.get("input", "")

            print(f"Step {step.get('step', '?')}: {action}({input_val})")

            if action in self.tools:
                if input_val:
                    result = self.tools[action](input_val)
                else:
                    result = self.tools[action]()
            else:
                result = f"Unknown action: {action}"

            results.append({
                "step": step.get("step"),
                "action": action,
                "result": result
            })
            print(f"  → {result}")

        return results

    def synthesize(self, task: str, results: list) -> str:
        """Synthesize final answer from results."""
        prompt = f"""Task: {task}

Results from executing the plan:
{json.dumps(results, indent=2)}

Provide a clear, concise answer based on these results."""

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content

    def run(self, task: str) -> str:
        """Plan, execute, and synthesize."""
        print("📋 Creating plan...")
        plan = self.create_plan(task)
        print(f"Plan: {len(plan)} steps\n")

        print("🔧 Executing plan...")
        results = self.execute_plan(plan)
        print()

        print("📝 Synthesizing answer...")
        answer = self.synthesize(task, results)

        return answer

# Test planning agent
planning_agent = PlanningAgent()
result = planning_agent.run("Search for Python and calculate 2^10")
print(f"\n✓ Answer: {result}")

## 4. Agent with Memory

In [ ]:
class MemoryAgent:
    """Agent with memory across conversations."""

    def __init__(self):
        self.client = OpenAI()
        self.tools = TOOLS
        self.memory = []  # Store key facts
        self.conversation = []

    def remember(self, fact: str):
        """Store a fact in memory."""
        self.memory.append(fact)
        print(f"  💾 Remembered: {fact}")

    def get_memory_context(self) -> str:
        """Get memory as context."""
        if not self.memory:
            return "No facts in memory."
        return "Known facts:\n" + "\n".join(f"- {m}" for m in self.memory[-10:])

    def chat(self, message: str) -> str:
        """Chat with memory and tool support."""

        system_prompt = f"""You are a helpful assistant with memory.

{self.get_memory_context()}

When you learn new facts, use [REMEMBER: fact] to store them.
"""

        self.conversation.append({"role": "user", "content": message})

        messages = [{"role": "system", "content": system_prompt}] + self.conversation

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages
        )

        reply = response.choices[0].message.content

        # Extract and store memories
        import re
        memories = re.findall(r'\[REMEMBER:\s*(.+?)\]', reply)
        for mem in memories:
            self.remember(mem)

        # Clean reply
        clean_reply = re.sub(r'\[REMEMBER:.+?\]', '', reply).strip()

        self.conversation.append({"role": "assistant", "content": clean_reply})

        return clean_reply

# Test memory agent
mem_agent = MemoryAgent()

print(mem_agent.chat("Hi! My name is Alice and I like Python."))
print()
print(mem_agent.chat("What's my name and what do I like?"))

## 5. Task Decomposition

In [ ]:
def decompose_task(task: str) -> list:
    """Break down a complex task into subtasks."""
    prompt = f"""Break down this task into smaller, actionable subtasks.

Task: {task}

Return JSON:
{{
    "subtasks": [
        {{"id": 1, "task": "description", "depends_on": []}},
        {{"id": 2, "task": "description", "depends_on": [1]}}
    ]
}}

Each subtask should be simple and specific."""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)["subtasks"]

# Test
complex_task = "Research Python trends and write a summary email"
subtasks = decompose_task(complex_task)

print(f"Task: {complex_task}\n")
print("Subtasks:")
for st in subtasks:
    deps = f" (depends on: {st['depends_on']})" if st['depends_on'] else ""
    print(f"  {st['id']}. {st['task']}{deps}")

## Summary

You learned:
- ✅ ReAct pattern (Reason → Act → Observe)
- ✅ OpenAI native tool agents
- ✅ Planning agents (plan → execute → synthesize)
- ✅ Agents with memory
- ✅ Task decomposition

**Next:** [Week 10 - Tool Calling](week_08_tool_calling.ipynb)